### Assignment 2 – Emotion Classification with a Graph Neural Network
- Discipline: EEE6553 – Advanced Deep Learning
- Task: Use a Graph Neural Network (GNN) to classify the emotions dataset (emotions-dataset.csv).
  Build a sample-similarity graph using cosine distance, implement custom message passing,
  and evaluate the classifier.
- Assignment: 2
- Student: Fabio Caversan

### 1. Libraries and Environment Setup
Import the required libraries and verify whether TensorFlow can use a GPU in the current notebook kernel.

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (
    classification_report, confusion_matrix, accuracy_score,
    precision_recall_fscore_support
)
from scipy.spatial.distance import cdist
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf

try:
    from tensorflow.keras import layers, Model
    _keras = tf.keras
except (ImportError, ModuleNotFoundError):
    import keras
    from keras import layers, Model
    _keras = keras

tf_version = tf.__version__
print("Python:", sys.version.split()[0])
print("TensorFlow:", tf_version)

gpus = tf.config.list_physical_devices("GPU")
print("GPUs detected:", gpus)
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    print("GPU is available. Training can use GPU.")
else:
    print("No GPU detected. Training will run on CPU.")
    print("For Windows GPU: switch kernel to 'Python (tf210-gpu)' – Python 3.10 + TF 2.10.")

## 2. Load and Explore the Emotions Dataset
Load the EEG emotions dataset, inspect its shape and column groups, and visualise the class distribution.

In [ ]:
candidate_files = [
    "datasets/emotions-dataset.csv",
    "./datasets/emotions-dataset.csv",
    "emotions-dataset.csv",
]
file_path = next((p for p in candidate_files if os.path.isfile(p)), None)
assert file_path is not None, "Dataset not found. Checked: " + ", ".join(candidate_files)

df = pd.read_csv(file_path)
print("Dataset path :", file_path)
print("Shape        :", df.shape)
print("Columns (first 10):", list(df.columns[:10]))
print("Label column :", df.columns[-1])
print("\nMissing values:", df.isna().sum().sum())
print("\nClass distribution:")
print(df["label"].value_counts())

## 3. Feature Preparation – Node Features
The dataset already contains 2 548 numeric features extracted from EEG signals (means, standard deviations, moments, covariance entries, eigenvalues, entropy, correlations, FFT coefficients – across two channels *a* and *b*).

Each **sample becomes a node** in the graph, and its 2 548-dimensional feature vector is standardised to zero mean / unit variance before being used as the node feature matrix **X**.

In [ ]:
feature_cols = df.columns[:-1]  # all columns except 'label'
X_raw = df[feature_cols].values.astype(np.float32)

scaler = StandardScaler()
X = scaler.fit_transform(X_raw).astype(np.float32)

num_nodes = X.shape[0]
num_features = X.shape[1]
print(f"Node feature matrix X: {num_nodes} nodes × {num_features} features")

## 4. Encode Emotion Labels
Convert the string labels (*NEGATIVE*, *NEUTRAL*, *POSITIVE*) into integer class indices with `LabelEncoder` and store the mapping for later interpretation.

In [ ]:
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(df["label"].values)
class_names = list(label_encoder.classes_)
num_classes = len(class_names)

print("Label mapping:", dict(zip(class_names, range(num_classes))))
print("Number of classes:", num_classes)
print("y shape:", y.shape)

## 5. Compute Pairwise Cosine Distance for Edge Construction
Compute the pairwise **cosine distance** between all node feature vectors. Cosine distance is well-suited for high-dimensional EEG feature spaces because it measures directional similarity regardless of magnitude. Smaller distance → more similar EEG patterns.

In [ ]:
cos_dist = cdist(X, X, metric="cosine").astype(np.float32)
cos_sim = 1.0 - cos_dist  # convert distance to similarity

print("Cosine distance matrix shape:", cos_dist.shape)
print(f"Distance range: [{cos_dist.min():.4f}, {cos_dist.max():.4f}]")
print(f"Similarity range: [{cos_sim.min():.4f}, {cos_sim.max():.4f}]")

## 6. Build the kNN Similarity Graph (Adjacency Matrix)
Create edges by connecting each node to its **k nearest neighbours** (based on cosine distance). The resulting adjacency matrix is row-normalised so that each node's incoming messages are averaged during message passing.

Using kNN instead of a global threshold keeps the graph sparse and ensures every node has exactly *k* connections, which stabilises GNN training.

In [ ]:
K = 10  # number of nearest neighbours per node

# For each node, find the k nearest neighbours (excluding itself)
A = np.zeros((num_nodes, num_nodes), dtype=np.float32)
for i in range(num_nodes):
    dists = cos_dist[i]
    # argsort returns indices of sorted distances; index 0 is self (dist=0), skip it
    knn_indices = np.argsort(dists)[1:K + 1]
    A[i, knn_indices] = 1.0

# Make the graph symmetric (undirected): if i→j or j→i, then both are connected
A = np.maximum(A, A.T)

# Add self-loops so each node retains its own features after aggregation
np.fill_diagonal(A, 1.0)

# Row-normalise so message passing computes a weighted average
row_sums = A.sum(axis=1, keepdims=True)
A_norm = A / row_sums

total_edges = int(A.sum()) - num_nodes  # subtract self-loops
avg_degree = total_edges / num_nodes

print(f"k = {K}")
print(f"Adjacency matrix shape: {A_norm.shape}")
print(f"Total edges (excl. self-loops): {total_edges}")
print(f"Average node degree: {avg_degree:.1f}")

## 7. Train / Test Split and Tensor Preparation
Create a **stratified 80 / 20 train / test split** using index masks (transductive setup – all nodes remain in the graph, but only training labels are used during optimisation). Convert all arrays to TensorFlow tensors.

In [ ]:
from sklearn.model_selection import train_test_split

train_idx, test_idx = train_test_split(
    np.arange(num_nodes), test_size=0.2, stratify=y, random_state=42
)

train_mask = np.zeros(num_nodes, dtype=np.float32)
test_mask = np.zeros(num_nodes, dtype=np.float32)
train_mask[train_idx] = 1.0
test_mask[test_idx] = 1.0

print(f"Train nodes: {int(train_mask.sum())} | Test nodes: {int(test_mask.sum())}")

# One-hot encode labels for cross-entropy
y_onehot = np.eye(num_classes, dtype=np.float32)[y]

# Convert to tensors (add batch dimension)
X_tensor = tf.expand_dims(tf.constant(X, dtype=tf.float32), axis=0)
A_tensor = tf.expand_dims(tf.constant(A_norm, dtype=tf.float32), axis=0)
y_tensor = tf.expand_dims(tf.constant(y_onehot, dtype=tf.float32), axis=0)
train_mask_tensor = tf.expand_dims(
    tf.constant(train_mask.reshape(-1, 1), dtype=tf.float32), axis=0
)
test_mask_tensor = tf.expand_dims(
    tf.constant(test_mask.reshape(-1, 1), dtype=tf.float32), axis=0
)

print("X_tensor shape :", X_tensor.shape)
print("A_tensor shape :", A_tensor.shape)
print("y_tensor shape :", y_tensor.shape)

## 8. Custom Message Passing Layer
Define a custom `MessagePassingLayer` that implements the core GNN operation:

1. **Message** – each node's features are transformed by a learnable MLP to produce messages.
2. **Aggregate** – messages from neighbours are aggregated via the normalised adjacency matrix (mean aggregation).
3. **Update** – the aggregated neighbourhood signal is concatenated with the node's own features and passed through an update MLP to produce new node representations.

In [ ]:
class MessagePassingLayer(layers.Layer):
    """Custom message-passing GNN layer.

    message():   MLP transforms node features into messages
    aggregate(): normalised adjacency matrix multiplies messages (mean aggregation)
    update():    concatenate [own_features, aggregated_messages] → MLP → new features
    """

    def __init__(self, hidden_units, dropout_rate=0.0, **kwargs):
        super().__init__(**kwargs)
        # Message function: transforms node features before aggregation
        self.message_mlp = _keras.Sequential([
            layers.Dense(hidden_units, activation="relu"),
            layers.Dense(hidden_units, activation="relu"),
        ])
        # Update function: combines own features with aggregated neighbourhood
        self.update_mlp = _keras.Sequential([
            layers.Dense(hidden_units, activation="relu"),
            layers.Dropout(dropout_rate),
            layers.Dense(hidden_units, activation="relu"),
        ])

    def call(self, inputs, training=False):
        X, A = inputs  # X: (batch, N, F), A: (batch, N, N)

        # 1. MESSAGE — produce messages from node features
        messages = self.message_mlp(X, training=training)        # (batch, N, H)

        # 2. AGGREGATE — mean neighbourhood aggregation via adjacency
        aggregated = tf.matmul(A, messages)                      # (batch, N, H)

        # 3. UPDATE — combine own representation with aggregated messages
        combined = tf.concat([X, aggregated], axis=-1)           # (batch, N, F+H)
        return self.update_mlp(combined, training=training)      # (batch, N, H)

print("MessagePassingLayer defined ✓")

## 9. GNN Classifier Architecture
Stack multiple message-passing layers with **residual connections** to build the full classifier:

1. **Input projection** — reduce the 2 548-dim features to the hidden dimension.
2. **Message-passing blocks** — 3 layers with residual shortcuts so gradients flow easily.
3. **Classification head** — Dense → Dropout → Dense(3, softmax) for the three emotion classes.

The model processes the entire graph in one forward pass (transductive learning).

In [ ]:
class GNNClassifier(Model):
    def __init__(self, hidden_units=128, num_gnn_layers=3, num_classes=3, dropout_rate=0.3):
        super().__init__()
        self.input_proj = layers.Dense(hidden_units, activation="relu")
        self.gnn_layers = [
            MessagePassingLayer(hidden_units, dropout_rate=dropout_rate)
            for _ in range(num_gnn_layers)
        ]
        self.classifier = _keras.Sequential([
            layers.Dense(hidden_units, activation="relu"),
            layers.Dropout(dropout_rate),
            layers.Dense(num_classes, activation="softmax"),
        ])

    def call(self, inputs, training=False):
        X, A = inputs
        H = self.input_proj(X)
        for gnn_layer in self.gnn_layers:
            H = H + gnn_layer([H, A], training=training)  # residual connection
        return self.classifier(H, training=training)

    def get_embeddings(self, inputs, training=False):
        """Return the node embeddings before the classification head."""
        X, A = inputs
        H = self.input_proj(X)
        for gnn_layer in self.gnn_layers:
            H = H + gnn_layer([H, A], training=training)
        return H

# Hyper-parameters
HIDDEN = 128
GNN_LAYERS = 3
DROPOUT = 0.3
LR = 0.001
EPOCHS = 150

model = GNNClassifier(
    hidden_units=HIDDEN,
    num_gnn_layers=GNN_LAYERS,
    num_classes=num_classes,
    dropout_rate=DROPOUT,
)
optimizer = _keras.optimizers.Adam(learning_rate=LR)

# Warm up the model to print summary
_ = model([X_tensor, A_tensor], training=False)
model.summary()

## 10. Training Loop
Train the GNN using masked cross-entropy loss (only training nodes contribute to the gradient). Track both training and test accuracy/loss every 10 epochs for later visualisation.

In [ ]:
def masked_crossentropy(y_true, y_pred, mask):
    """Cross-entropy loss applied only to masked (selected) nodes."""
    loss = -tf.reduce_sum(y_true * tf.math.log(y_pred + 1e-8), axis=-1, keepdims=True)
    loss = loss * mask
    return tf.reduce_sum(loss) / (tf.reduce_sum(mask) + 1e-8)

def masked_accuracy(y_true, y_pred, mask):
    """Accuracy computed only on masked nodes."""
    correct = tf.cast(
        tf.equal(tf.argmax(y_true, axis=-1), tf.argmax(y_pred, axis=-1)),
        tf.float32,
    )
    correct = correct * mask[:, :, 0]
    return tf.reduce_sum(correct) / (tf.reduce_sum(mask[:, :, 0]) + 1e-8)

history = {"train_loss": [], "test_loss": [], "train_acc": [], "test_acc": []}

for epoch in range(EPOCHS):
    with tf.GradientTape() as tape:
        y_pred = model([X_tensor, A_tensor], training=True)
        loss = masked_crossentropy(y_tensor, y_pred, train_mask_tensor)

    grads = tape.gradient(loss, model.trainable_variables)
    optimizer.apply_gradients(zip(grads, model.trainable_variables))

    # Track metrics
    train_acc = masked_accuracy(y_tensor, y_pred, train_mask_tensor).numpy()

    y_pred_eval = model([X_tensor, A_tensor], training=False)
    test_loss = masked_crossentropy(y_tensor, y_pred_eval, test_mask_tensor).numpy()
    test_acc = masked_accuracy(y_tensor, y_pred_eval, test_mask_tensor).numpy()

    history["train_loss"].append(loss.numpy())
    history["test_loss"].append(test_loss)
    history["train_acc"].append(train_acc)
    history["test_acc"].append(test_acc)

    if (epoch + 1) % 30 == 0 or epoch == 0:
        print(
            f"Epoch {epoch + 1:03d} | "
            f"Train Loss: {loss.numpy():.4f}  Acc: {train_acc:.4f} | "
            f"Test  Loss: {test_loss:.4f}  Acc: {test_acc:.4f}"
        )

## 11. Evaluate Model Performance
Evaluate the trained GNN on the held-out test nodes. Print accuracy, precision, recall, F1-score, and the full per-class classification report.

In [ ]:
y_pred_final = model([X_tensor, A_tensor], training=False).numpy()[0]

y_pred_classes = np.argmax(y_pred_final, axis=1)[test_idx]
y_true_classes = y[test_idx]

test_accuracy = accuracy_score(y_true_classes, y_pred_classes)
precision, recall, f1, _ = precision_recall_fscore_support(
    y_true_classes, y_pred_classes, average="weighted"
)

print("=" * 55)
print("       GNN EMOTION CLASSIFICATION — TEST RESULTS")
print("=" * 55)
print(f"  Accuracy  : {test_accuracy:.4f}")
print(f"  Precision : {precision:.4f}  (weighted)")
print(f"  Recall    : {recall:.4f}  (weighted)")
print(f"  F1-Score  : {f1:.4f}  (weighted)")
print("=" * 55)

print("\n=== PER-CLASS CLASSIFICATION REPORT ===")
print(classification_report(y_true_classes, y_pred_classes,
                            target_names=class_names, digits=4))

print("=== CONFUSION MATRIX ===")
cm = confusion_matrix(y_true_classes, y_pred_classes)
print(cm)

## 12. Visualise Training Metrics and Confusion Matrix
Plot training/test loss and accuracy curves, and display the confusion matrix as a heatmap.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# --- Loss curves ---
axes[0].plot(history["train_loss"], label="Train Loss")
axes[0].plot(history["test_loss"], label="Test Loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].set_title("Loss Curves")
axes[0].legend()
axes[0].grid(True)

# --- Accuracy curves ---
axes[1].plot(history["train_acc"], label="Train Accuracy")
axes[1].plot(history["test_acc"], label="Test Accuracy")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy")
axes[1].set_title("Accuracy Curves")
axes[1].legend()
axes[1].grid(True)

# --- Confusion matrix heatmap ---
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=class_names, yticklabels=class_names, ax=axes[2])
axes[2].set_xlabel("Predicted")
axes[2].set_ylabel("True")
axes[2].set_title("Confusion Matrix")

plt.tight_layout()
plt.show()

## 13. Visualise Node Embeddings with t-SNE
Extract the learned node embeddings from the GNN (before the classification head) and project them into 2D using t-SNE. Nodes are coloured by their true emotion label to assess how well the GNN separates the classes in its learned representation space.

In [ ]:
from sklearn.manifold import TSNE

embeddings = model.get_embeddings([X_tensor, A_tensor], training=False).numpy()[0]

tsne = TSNE(n_components=2, random_state=42, perplexity=30)
emb_2d = tsne.fit_transform(embeddings)

plt.figure(figsize=(10, 7))
colours = ["#e74c3c", "#3498db", "#2ecc71"]
for idx, name in enumerate(class_names):
    mask = y == idx
    plt.scatter(emb_2d[mask, 0], emb_2d[mask, 1],
                label=name, alpha=0.6, s=15, c=colours[idx])

plt.legend(fontsize=12)
plt.title("t-SNE of GNN Node Embeddings (coloured by true emotion)")
plt.xlabel("t-SNE 1")
plt.ylabel("t-SNE 2")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()